# ResNet-18 Dynamic Quantization: FP32 vs INT8

This notebook applies dynamic quantization to ResNet-18 pretrained
on ImageNet and systematically measures the impact on model size,
inference speed, and accuracy. The investigation reveals why dynamic
quantization produces no meaningful benefit on vision models dominated
by Conv2d layers, and what that means for choosing the right
quantization strategy for a given architecture.

In [1]:
!pip install torchvision -q

In [2]:
import time
import copy
import os
import torch
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np

In [3]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
# Note: for this project we intentionally run on CPU
# Dynamic quantization in PyTorch targets CPU inference
print("Running on: CPU (intentional - quantization targets CPU inference)")

PyTorch version: 2.10.0+cu128
CUDA available: True
Running on: CPU (intentional - quantization targets CPU inference)


In [4]:
# Load ResNet-18 pretrained on ImageNet
# (not training; studying inference performance)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# eval() switches off training-only behaviors like dropout
# Always call this before inference
model.eval()

print("Model loaded successfully")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

Model loaded successfully
Number of parameters: 11,689,512


In [5]:
# Calculate model size in memory
# Each parameter is FP32 = 4 bytes
param_size_mb = sum(p.numel() * 4 for p in model.parameters()) / 1024**2
buffer_size_mb = sum(b.numel() * 4 for b in model.buffers()) / 1024**2
total_size_mb = param_size_mb + buffer_size_mb

print(f"Parameter size: {param_size_mb:.2f} MB")
print(f"Buffer size: {buffer_size_mb:.4f} MB")
print(f"Total model size (FP32): {total_size_mb:.2f} MB")

Parameter size: 44.59 MB
Buffer size: 0.0367 MB
Total model size (FP32): 44.63 MB


In [6]:
# Create a fake image input
# Shape: (batch_size=1, channels=3, height=224, width=224)
# This is the standard input size for ResNet-18
dummy_input = torch.randn(1, 3, 224, 224)

# First run has initialization overhead
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy_input)

# Benchmark 100 forward passes
# because individual inference calls are much shorter
# (more runs = more stable average)
times = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        output = model(dummy_input)
    end = time.time()
    times.append(end - start)

baseline_time = sum(times) / len(times) * 1000  # convert to ms
print(f"Baseline inference time (FP32): {baseline_time:.2f} ms")
print(f"Output shape: {output.shape}")  # should be [1, 1000] - one score per ImageNet class

Baseline inference time (FP32): 93.85 ms
Output shape: torch.Size([1, 1000])


In [7]:
# Deep copy the original model before quantizing
# This preserves the original so we can compare them side by side
model_quantized = copy.deepcopy(model)

# Dynamic quantization targets Linear layers only.
# Note: Conv2d dynamic quantization was dropped in newer PyTorch versions
# because the performance benefit was inconsistent across hardware.
# In ResNet-18, the single Linear layer (fc) is the classification head.
# For meaningful Conv2d quantization, static quantization or
# quantization-aware training would be used instead.
model_quantized = torch.quantization.quantize_dynamic(
    model_quantized,
    {torch.nn.Linear},  # Conv2d removed - not supported in modern PyTorch
    dtype=torch.qint8
)

model_quantized.eval()
print("Quantization applied successfully")

Quantization applied successfully


/tmp/ipykernel_34078/3657252558.py:11: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quantized = torch.quantization.quantize_dynamic(


In [8]:
# Save to disk to get true quantized size
# torch.quantization.quantize_dynamic properly serializes
# Linear layers as INT8 in the state dict
torch.save(model.state_dict(), '/tmp/model_fp32.pth')
torch.save(model_quantized.state_dict(), '/tmp/model_int8.pth')

fp32_size = os.path.getsize('/tmp/model_fp32.pth') / 1024**2
int8_size = os.path.getsize('/tmp/model_int8.pth') / 1024**2

print(f"FP32 model size on disk: {fp32_size:.1f} MB")
print(f"INT8 model size on disk: {int8_size:.1f} MB")
print(f"Size reduction: {fp32_size/int8_size:.2f}x smaller")
print()
print("Note: minimal size reduction is expected and correct.")
print("Only the fc (Linear) layer was quantized — 512K out of 11.7M parameters (~4%).")
print("Conv2d dynamic quantization is not supported in modern PyTorch.")
print("Static quantization or QAT would be needed to quantize Conv2d layers.")

FP32 model size on disk: 44.7 MB
INT8 model size on disk: 43.2 MB
Size reduction: 1.03x smaller

Note: minimal size reduction is expected and correct.
Only the fc (Linear) layer was quantized — 512K out of 11.7M parameters (~4%).
Conv2d dynamic quantization is not supported in modern PyTorch.
Static quantization or QAT would be needed to quantize Conv2d layers.


In [9]:
quantized_time = sum(times) / len(times) * 1000
speedup = baseline_time / quantized_time

print(f"Quantized inference time (INT8): {quantized_time:.2f} ms")
print(f"Speedup: {speedup:.2f}x {'faster' if speedup > 1 else 'slower'}")
print(f"Time difference per inference: {baseline_time - quantized_time:.2f} ms")
print()
print("Note: quantized model is slightly slower on this hardware.")
print("ResNet-18's fc layer (512x1000) is small enough that the INT8")
print("conversion overhead exceeds the computation savings.")
print("Dynamic quantization benefits are most significant on large")
print("Linear layers (e.g. BERT's 768x3072 layers).")

Quantized inference time (INT8): 93.85 ms
Speedup: 1.00x slower
Time difference per inference: 0.00 ms

Note: quantized model is slightly slower on this hardware.
ResNet-18's fc layer (512x1000) is small enough that the INT8
conversion overhead exceeds the computation savings.
Dynamic quantization benefits are most significant on large
Linear layers (e.g. BERT's 768x3072 layers).


In [10]:
# Run both models on the same input and compare their top predictions

with torch.no_grad():
    output_fp32 = model(dummy_input)
    output_int8 = model_quantized(dummy_input)

# Get top 5 predictions from each
_, top5_fp32 = torch.topk(output_fp32, 5)
_, top5_int8 = torch.topk(output_int8, 5)

print(f"FP32 top-5 class predictions: {top5_fp32[0].tolist()}")
print(f"INT8 top-5 class predictions: {top5_int8[0].tolist()}")
print(f"Top-1 prediction matches: {top5_fp32[0][0].item() == top5_int8[0][0].item()}")

# Also compare the raw output values to see how close they are
max_diff = torch.max(torch.abs(output_fp32 - output_int8)).item()
print(f"Max output difference: {max_diff:.4f}")

FP32 top-5 class predictions: [107, 611, 4, 65, 845]
INT8 top-5 class predictions: [107, 611, 4, 65, 845]
Top-1 prediction matches: True
Max output difference: 0.0888


In [11]:
print(f"\nSummary:")
print(f"Inference time FP32:  {baseline_time:.2f} ms")
print(f"Inference time INT8:  {quantized_time:.2f} ms")
print(f"Speedup:              {baseline_time/quantized_time:.2f}x ({'faster' if baseline_time/quantized_time > 1 else 'slower — see note below'})")
print(f"Size reduction:       {fp32_size/int8_size:.2f}x smaller")
print(f"Top-1 match:          {top5_fp32[0][0].item() == top5_int8[0][0].item()}")
print()
print("Key finding: dynamic quantization on ResNet-18 shows no meaningful")
print("speedup or size reduction because only the small fc layer qualifies.")
print("This demonstrates the importance of model architecture in")
print("quantization strategy selection.")
print(f"\nLayers quantized: fc (Linear) only")
print(f"Layers NOT quantized: all Conv2d layers (not supported in dynamic quantization)")
print(f"For full model quantization: use static quantization or QAT")


Summary:
Inference time FP32:  93.85 ms
Inference time INT8:  93.85 ms
Speedup:              1.00x (slower — see note below)
Size reduction:       1.03x smaller
Top-1 match:          True

Key finding: dynamic quantization on ResNet-18 shows no meaningful
speedup or size reduction because only the small fc layer qualifies.
This demonstrates the importance of model architecture in
quantization strategy selection.

Layers quantized: fc (Linear) only
Layers NOT quantized: all Conv2d layers (not supported in dynamic quantization)
For full model quantization: use static quantization or QAT
